<a href="https://colab.research.google.com/github/Ruhul73/Cats_and_dogs_prediction/blob/main/Cats_and_dogs_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import tarfile
import urllib.request
from pathlib import Path

# URLs define karna
DATA_URLS = {
    "images": "https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz",
    "annotations": "https://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz"
}

# Current directory set karna
BASE_DIR = Path(".")

def download_and_extract(name, url, base_path):
    tar_path = base_path / f"{name}.tar.gz"
    extract_path = base_path / name  # e.g., './images'

    # Check 1: Agar folder pehle se hai to download aur extract dono skip karo
    if extract_path.exists() and extract_path.is_dir():
        print(f"✅ '{name}' directory already exists. Skipping download!")
        return

    # Check 2: Agar tar file nahi hai to download karo
    if not tar_path.exists():
        print(f"⬇️ Downloading {name} from Oxford servers... (Please wait)")
        urllib.request.urlretrieve(url, tar_path)

    # Extract karna
    print(f"📦 Extracting {name}...")
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(path=base_path)

    print(f"✅ {name.capitalize()} successfully downloaded and extracted!")

# Process both datasets
for name, url in DATA_URLS.items():
    download_and_extract(name, url, BASE_DIR)

# Strictly count ONLY image files (.jpg) using Pathlib
image_dir = BASE_DIR / 'images'
if image_dir.exists():
    # .glob() use karke sirf .jpg files count karna
    image_files = list(image_dir.glob("*.jpg"))
    print(f"📸 Total valid images found: {len(image_files)}")
else:
    print("⚠️ Images folder not found!")

⬇️ Downloading images from Oxford servers... (Please wait)
📦 Extracting images...


/tmp/ipykernel_1103/2846102590.py:32: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=base_path)


✅ Images successfully downloaded and extracted!
⬇️ Downloading annotations from Oxford servers... (Please wait)
📦 Extracting annotations...
✅ Annotations successfully downloaded and extracted!
📸 Total valid images found: 7390


# Custom Data Loader, Breed Dictionary & Model Initialization

In [2]:
import os
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from pathlib import Path
from sklearn.model_selection import train_test_split

# 1. Map Breeds to Species (Cat/Dog) & Characteristics (As provided)
BREED_TRAITS = {
    'abyssinian': {'species': 'Cat', 'traits': 'Short hair, highly active, large pointed ears, curious nature.'},
    'bengal': {'species': 'Cat', 'traits': 'Spotted coat looks like a leopard, very energetic, loves water.'},
    'birman': {'species': 'Cat', 'traits': 'Long hair, pale body with dark points, deep blue eyes, gentle.'},
    'bombay': {'species': 'Cat', 'traits': 'Glossy black coat, copper eyes, affectionate, panther-like.'},
    'british_shorthair': {'species': 'Cat', 'traits': 'Chunky body, dense coat, calm and easygoing.'},
    'egyptian_mau': {'species': 'Cat', 'traits': 'Naturally spotted, extremely fast runner, very loyal.'},
    'maine_coon': {'species': 'Cat', 'traits': 'Massive size, tufted ears, bushy tail, gentle giant.'},
    'persian': {'species': 'Cat', 'traits': 'Flat face, long flowing coat, sweet and quiet personality.'},
    'ragdoll': {'species': 'Cat', 'traits': 'Goes limp when picked up, bright blue eyes, very docile.'},
    'russian_blue': {'species': 'Cat', 'traits': 'Dense silvery-blue coat, green eyes, reserved but affectionate.'},
    'siamese': {'species': 'Cat', 'traits': 'Sleek body, striking color points, extremely vocal and social.'},
    'sphynx': {'species': 'Cat', 'traits': 'Hairless, wrinkled skin, large ears, very attention-seeking.'},
    'american_bulldog': {'species': 'Dog', 'traits': 'Stocky, muscular, short coat, confident and athletic.'},
    'american_pit_bull_terrier': {'species': 'Dog', 'traits': 'Medium-sized, solid build, highly eager to please.'},
    'basset_hound': {'species': 'Dog', 'traits': 'Short legs, very long ears, excellent sense of smell.'},
    'beagle': {'species': 'Dog', 'traits': 'Small hound, incredibly strong sense of smell, merry and vocal.'},
    'boxer': {'species': 'Dog', 'traits': 'Muscular, square jaw, playful, highly energetic and protective.'},
    'chihuahua': {'species': 'Dog', 'traits': 'Tiny size, large round eyes, big personality, fiercely loyal.'},
    'english_cocker_spaniel': {'species': 'Dog', 'traits': 'Medium size, floppy ears, happy and energetic bird dog.'},
    'english_setter': {'species': 'Dog', 'traits': 'Speckled coat (belton), elegant, gentle and mellow.'},
    'german_shorthaired': {'species': 'Dog', 'traits': 'Sleek, powerful, highly energetic hunting dog.'},
    'great_pyrenees': {'species': 'Dog', 'traits': 'Huge, thick white coat, calm but fiercely protective guardian.'},
    'havanese': {'species': 'Dog', 'traits': 'Small, long silky hair, springy step, very sociable.'},
    'japanese_chin': {'species': 'Dog', 'traits': 'Tiny, flat face, luxurious mane, cat-like climbing ability.'},
    'keeshond': {'species': 'Dog', 'traits': 'Plush grey/black coat, "spectacles" around eyes, very friendly.'},
    'leonberger': {'species': 'Dog', 'traits': 'Giant size, lion-like mane, surprisingly gentle and patient.'},
    'miniature_pinscher': {'species': 'Dog', 'traits': 'Small but sturdy, high-stepping gait, fearless.'},
    'newfoundland': {'species': 'Dog', 'traits': 'Massive, thick water-resistant coat, excellent swimmer, sweet.'},
    'pomeranian': {'species': 'Dog', 'traits': 'Tiny fluffy ball, foxy face, bold and lively.'},
    'pug': {'species': 'Dog', 'traits': 'Compact, deep wrinkles, flat face, charming and loving.'},
    'saint_bernard': {'species': 'Dog', 'traits': 'Giant, powerful, famous rescue dog, incredibly patient.'},
    'samoyed': {'species': 'Dog', 'traits': 'Fluffy white coat, "Sammy smile", friendly and cold-tolerant.'},
    'scottish_terrier': {'species': 'Dog', 'traits': 'Distinctive beard and eyebrows, independent and feisty.'},
    'shiba_inu': {'species': 'Dog', 'traits': 'Fox-like appearance, curly tail, very clean and independent.'},
    'staffordshire_bull_terrier': {'species': 'Dog', 'traits': 'Muscular, broad head, courageous but very affectionate.'},
    'wheaten_terrier': {'species': 'Dog', 'traits': 'Soft silky wavy coat, deeply devoted, jumps to greet.'},
    'yorkshire_terrier': {'species': 'Dog', 'traits': 'Tiny, long silky hair, feisty, brave and bossy.'}
}

# 2. Optimized PyTorch Dataset
class OxfordPetDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = Path(image_dir)
        self.transform = transform

        self.image_files = []
        self.labels = []
        self.classes = sorted(list(BREED_TRAITS.keys())) # Use dict keys as source of truth

        # Parse files efficiently
        for img_path in self.image_dir.glob('*.jpg'):
            filename = img_path.name
            # Regex to extract breed name before the last underscore and number
            match = re.match(r'(.+)_\d+\.jpg$', filename)
            if match:
                breed = match.group(1).lower()
                if breed in self.classes:
                    self.image_files.append(img_path)
                    self.labels.append(self.classes.index(breed))

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        try:
            # Force convert to RGB
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            # Return a dummy tensor if image is corrupt to avoid crashing
            image = Image.new('RGB', (224, 224))

        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

# 3. Improved Data Transformations (Training with Augmentation)
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 4. Initialize Dataset
print("Processing dataset...")
# Using val_transforms here just to initialize and check counts.
# During actual training loop, you should split train/val sets and apply train_transforms to training set.
full_dataset = OxfordPetDataset(image_dir='images', transform=val_transforms)
NUM_CLASSES = len(full_dataset.classes)

print(f"Successfully configured for {NUM_CLASSES} distinct breeds!")
print(f"Total valid images loaded: {len(full_dataset)}")

# 5. Initialize High-Level ResNet50 Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device.type.upper()}")

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
num_ftrs = model.fc.in_features
# Replace final layer
model.fc = nn.Linear(num_ftrs, NUM_CLASSES)
model = model.to(device)

print("✅ Data Engineering and Model Initialization Complete! Ready for the AI brain.")

Processing dataset...
Successfully configured for 37 distinct breeds!
Total valid images loaded: 7390
Using compute device: CUDA
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 198MB/s]


✅ Data Engineering and Model Initialization Complete! Ready for the AI brain.


In [3]:
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader, Subset
import time
import torch

print("Setting up DataLoaders for Training...")

# 1. FIXING THE TRANSFORM BUG: Alag alag subsets banayein taake Train aur Val ke transforms alag hon
dataset_size = len(full_dataset)
indices = torch.randperm(dataset_size).tolist()
train_size = int(0.8 * dataset_size)

# Do alag instances banayein taake sahi transforms apply hon
train_data_raw = OxfordPetDataset(image_dir='images', transform=train_transforms)
val_data_raw = OxfordPetDataset(image_dir='images', transform=val_transforms)

# Subset banayein indices ki base par
train_dataset = Subset(train_data_raw, indices[:train_size])
val_dataset = Subset(val_data_raw, indices[train_size:])

# pin_memory=True add kiya for faster CPU to GPU transfer
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Training on {len(train_dataset)} images, Validating on {len(val_dataset)} images.")

# 2. Cleaner Freezing Logic
# Pehle existing base model ke sab weights freeze karein
for param in model.parameters():
    param.requires_grad = False

# Ab final layer replace karein (Yeh by default requires_grad=True hogi)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, NUM_CLASSES).to(device)

# 3. Loss Function, Optimizer & Scheduler
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# Scheduler: Har 2 epochs ke baad learning rate ko 0.1 se multiply kar dega (Decay)
exp_lr_scheduler = lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.1)

# 4. The Training Loop
num_epochs = 5
best_acc = 0.0

print("\n🚀 Starting Model Training...\n")

for epoch in range(num_epochs):
    print(f'Epoch {epoch+1}/{num_epochs} | LR: {optimizer.param_groups[0]["lr"]:.6f}')
    print('-' * 20)

    since = time.time()

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
            dataloader = train_loader
        else:
            model.eval()
            dataloader = val_loader

        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            # Batch loss aur accuracy accumulate karein
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        # Scheduler ko sirf training phase ke baad step karna hota hai
        if phase == 'train':
            exp_lr_scheduler.step()

        epoch_loss = running_loss / len(dataloader.dataset)
        epoch_acc = running_corrects.double() / len(dataloader.dataset)

        print(f'{phase.capitalize():>5} -> Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.4f}')

        # Best model logic
        if phase == 'val' and epoch_acc > best_acc:
            best_acc = epoch_acc
            torch.save(model.state_dict(), 'best_pet_model.pth')
            print("         🌟 New best model saved!")

    time_elapsed = time.time() - since
    print(f'Time: {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s\n')

print(f'✅ Training Complete! Best Validation Accuracy: {best_acc:.4f}')

# Best weights reload karein
model.load_state_dict(torch.load('best_pet_model.pth'))
model.eval()
print("Ready for Inference!")

Setting up DataLoaders for Training...
Training on 5912 images, Validating on 1478 images.

🚀 Starting Model Training...

Epoch 1/5 | LR: 0.001000
--------------------
Train -> Loss: 1.3697 | Accuracy: 0.6617
  Val -> Loss: 0.3674 | Accuracy: 0.9026
         🌟 New best model saved!
Time: 0m 47s

Epoch 2/5 | LR: 0.001000
--------------------
Train -> Loss: 0.6818 | Accuracy: 0.8023
  Val -> Loss: 0.3076 | Accuracy: 0.9114
         🌟 New best model saved!
Time: 0m 45s

Epoch 3/5 | LR: 0.000100
--------------------
Train -> Loss: 0.5354 | Accuracy: 0.8464
  Val -> Loss: 0.2715 | Accuracy: 0.9235
         🌟 New best model saved!
Time: 0m 48s

Epoch 4/5 | LR: 0.000100
--------------------
Train -> Loss: 0.5172 | Accuracy: 0.8520
  Val -> Loss: 0.2618 | Accuracy: 0.9269
         🌟 New best model saved!
Time: 0m 47s

Epoch 5/5 | LR: 0.000010
--------------------
Train -> Loss: 0.5010 | Accuracy: 0.8577
  Val -> Loss: 0.2612 | Accuracy: 0.9269
Time: 0m 48s

✅ Training Complete! Best Validation

In [4]:
# 1. Required Libraries Install karein
!pip install -q grad-cam gradio

import gradio as gr
import numpy as np
import torch
import torch.nn.functional as F
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

# 🧹 Purane hidden servers band karein
gr.close_all()

print("Setting up the Ultimate FYP-Ready Pet Classifier...")

# 2. Model Preparation & XAI Fix
model.eval()

# 🔥 THE FIX: Grad-CAM ke liye target layer ko "Unfreeze" karein (NoneType error se bachne ke liye)
for param in model.layer4[-1].parameters():
    param.requires_grad = True

# 3. Setup Grad-CAM
target_layers = [model.layer4[-1]]
cam = GradCAM(model=model, target_layers=target_layers)

CONFIDENCE_THRESHOLD = 0.30

def predict_and_explain(image_pil):
    try:
        if image_pil is None:
            return "Please upload an image.", None

        # Ensure image is strictly RGB
        image_pil = image_pil.convert('RGB')

        # Preprocess
        input_tensor = val_transforms(image_pil).unsqueeze(0).to(device)

        # Predict
        with torch.no_grad():
            outputs = model(input_tensor)
            probabilities = F.softmax(outputs, dim=1)

            # 🔥 THE FIX 2: Top 3 predictions nikal rahe hain!
            top_probs, top_classes = torch.topk(probabilities, 3, dim=1)

            # Tensors ko lists mein convert kar rahe hain
            top_probs = top_probs[0].tolist()
            top_classes = top_classes[0].tolist()

        # Top-1 Prediction (Primary)
        primary_prob = top_probs[0]
        primary_class_idx = top_classes[0]

        # 🛑 OOD Check (Agar Top guess bhi 30% se kam ho)
        if primary_prob < CONFIDENCE_THRESHOLD:
            warning_msg = (f"⚠️ UNRECOGNIZED IMAGE DETECTED!\n\n"
                           f"Highest Confidence: {primary_prob*100:.2f}%\n"
                           f"This does not look like any Cat or Dog breed I know confidently.\n"
                           f"Please upload a clear picture of a pet.")
            return warning_msg, image_pil

        # ✅ Data Fetching & Formatting

        # 🥇 1st Place
        breed_1 = full_dataset.classes[primary_class_idx]
        info_1 = BREED_TRAITS.get(breed_1, {'species': 'Unknown', 'traits': 'No specific traits found.'})

        # 🔥 THE FIX 3: Traits ko proper Bullet Points mein convert karna
        traits_list = info_1['traits'].split(',')
        formatted_traits = "\n".join([f"  • {trait.strip().capitalize()}" for trait in traits_list])

        # 🥈 2nd Place
        breed_2 = full_dataset.classes[top_classes[1]]
        prob_2 = top_probs[1]

        # 🥉 3rd Place
        breed_3 = full_dataset.classes[top_classes[2]]
        prob_3 = top_probs[2]

        # 🎨 Generate Explainability Heatmap
        grayscale_cam = cam(input_tensor=input_tensor, targets=None)[0, :]
        img_resized = image_pil.resize((224, 224))
        img_float = np.float32(img_resized) / 255
        cam_image = show_cam_on_image(img_float, grayscale_cam, use_rgb=True)

        # ✨ Final Result Text Format
        result_text = (f"🥇 Top Prediction: {breed_1.replace('_', ' ').title()} ({primary_prob*100:.2f}%)\n"
                       f"🐾 Species Category: {info_1['species']}\n\n"
                       f"🧬 Key Characteristics:\n{formatted_traits}\n\n"
                       f"---------------------------\n"
                       f"🤔 AI's Other Guesses (Confusion Board):\n"
                       f"🥈 2nd Guess: {breed_2.replace('_', ' ').title()} ({prob_2*100:.2f}%)\n"
                       f"🥉 3rd Guess: {breed_3.replace('_', ' ').title()} ({prob_3*100:.2f}%)\n\n"
                       f"---------------------------\n"
                       f"🧠 [Architecture Note]: The Heatmap on the right shows exactly which physical features the Neural Network looked at to make this decision.")

        return result_text, cam_image

    except Exception as e:
        error_msg = f"❌ CODE CRASHED!\n\nError Name: {str(e)}\n\n(Please check if you ran the earlier Data & Training cells!)"
        return error_msg, None

# 4. Create Attractive Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("<center><h1> 🐶🐱 Ultimate Pet Classifier (FYP Edition) </h1></center>")
    gr.Markdown("<center>Top-3 Predictions + Bulleted Characteristics + Explainable AI (Heatmap)</center>")

    with gr.Row():
        with gr.Column():
            input_image = gr.Image(type="pil", label="Upload Pet Image Here")
            submit_btn = gr.Button("Analyze Pet 🚀", variant="primary")

        with gr.Column():
            output_text = gr.Textbox(label="AI Analysis Results", lines=14)
            output_heatmap = gr.Image(label="XAI Heatmap (Grad-CAM)")

    submit_btn.click(
        fn=predict_and_explain,
        inputs=input_image,
        outputs=[output_text, output_heatmap]
    )

print("✅ GUI is Ready! Running the web app below...")
demo.launch(share=True, debug=False)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 113.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Setting up the Ultimate FYP-Ready Pet Classifier...
✅ GUI is Ready! Running the web app below...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9d6b0489d0c16bbf7a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
